In [ ]:
import os
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("default")

dataset_root = Path("/kaggle/input/weed-detection")
working_root = Path("/kaggle/working")
if not dataset_root.is_dir():
    raise FileNotFoundError(f"Expected the Kaggle dataset at {dataset_root}")
working_root.mkdir(parents=True, exist_ok=True)
file_count = sum(len(filenames) for _, _, filenames in os.walk(dataset_root))
print(f"Number of files in the input folder: {file_count}")
print(f"Working folder is ready: {working_root}")


In [ ]:
from PIL import Image

folder_path = '/kaggle/input/weed-detection/train'
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in {folder_path}: {file_count}")

source_dir = "/kaggle/input/weed-detection/train"
destination_dir = "/kaggle/working/"
os.makedirs(destination_dir, exist_ok=True)

for root, _, files in os.walk(source_dir):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            with Image.open(file_path) as img:
                img.verify()  
        except (IOError, SyntaxError):
            shutil.copy(file_path, destination_dir)
            print(f"Copied non-image file: {file_path}")

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

if not os.path.exists(source_dir):
    print(f"Source directory '{source_dir}' does not exist.")
else:
    image_files = [f for f in os.listdir(source_dir) if os.path.isfile(os.path.join(source_dir, f))]

    num_images = min(9, len(image_files))
    
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    fig.suptitle("Sample Images from Dataset", fontsize=16, fontweight='bold', color='darkblue')

    for i, ax in enumerate(axes.flat):
        if i < num_images:
            img_path = os.path.join(source_dir, image_files[i])
            try:
                with Image.open(img_path) as img:
                    ax.imshow(img)
                    ax.axis("off")  
                    ax.set_frame_on(True)  
            except (IOError, SyntaxError):
                ax.axis("off")
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.92) 
    plt.show()

In [ ]:
import json
from tqdm import tqdm

annotations_file = "/kaggle/working/_annotations.coco.json"
with open(annotations_file, 'r') as f:
    coco_data = json.load(f)

categories = coco_data.get("categories", [])
num_classes = len(categories)
class_names = [category['name'] for category in categories]
print(f"Number of Classes: {num_classes}")
print("Class Names:", class_names)

images_dir = "/kaggle/input/weed-detection/train"
output_dir = "/kaggle/working/yolo_format"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/images", exist_ok=True)
os.makedirs(f"{output_dir}/labels", exist_ok=True)

category_id_to_class_id = {cat["id"]: idx for idx, cat in enumerate(categories)}
errors = []

for annotation in tqdm(coco_data["annotations"], desc="Converting annotations"):
    try:
        image_id = annotation["image_id"]
        bbox = annotation["bbox"]
        category_id = annotation["category_id"]
        image_info = next(img for img in coco_data["images"] if img["id"] == image_id)
        img_filename = image_info["file_name"]
        img_width = image_info["width"]
        img_height = image_info["height"]

        x_min, y_min, width, height = bbox
        x_center = (x_min + width / 2) / img_width
        y_center = (y_min + height / 2) / img_height
        norm_width = width / img_width
        norm_height = height / img_height

        class_id = category_id_to_class_id[category_id]
        yolo_line = f"{class_id} {x_center} {y_center} {norm_width} {norm_height}\n"

        txt_filename = os.path.splitext(img_filename)[0] + ".txt"
        with open(os.path.join(output_dir, "labels", txt_filename), "a") as f:
            f.write(yolo_line)

        shutil.copy(os.path.join(images_dir, img_filename), os.path.join(output_dir, "images", img_filename))
    except Exception as e:
        errors.append((annotation, str(e)))

print(f"Conversion completed with {len(errors)} errors.")

In [ ]:
import os
import cv2
import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import random
import math

transformations = [
    A.Compose([A.HorizontalFlip(p=1.0)], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'])),  # انعكاس أفقي
    A.Compose([A.RandomRotate90(p=1.0)], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'])),   # تدوير 90 درجة
    A.Compose([A.RandomBrightnessContrast(p=1.0)], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'])),  # تغيير السطوع والتباين
    A.Compose([A.RandomScale(scale_limit=0.2, p=1.0)], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))  # تكبير عشوائي
]

images_dir = '/kaggle/working/yolo_format/images'
labels_dir = '/kaggle/working/yolo_format/labels'
output_images_dir = '/kaggle/working/yolo_format/augmented_images'
output_labels_dir = '/kaggle/working/yolo_format/augmented_labels'

os.makedirs(output_images_dir, exist_ok=True)
os.makedirs(output_labels_dir, exist_ok=True)

image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg') or f.endswith('.png')]

all_images = []  
for image_file in image_files:
    image_path = os.path.join(images_dir, image_file)
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 

    label_file = os.path.splitext(image_file)[0] + '.txt'
    label_path = os.path.join(labels_dir, label_file)
    
    with open(label_path, 'r') as f:
        lines = f.readlines()

    bboxes = []
    class_labels = []
    for line in lines:
        class_id, x_center, y_center, width, height = map(float, line.strip().split())
        bboxes.append([x_center, y_center, width, height])
        class_labels.append(class_id)

    all_images.append((image, bboxes, "Original"))

    for i, transform in enumerate(transformations):
        transformed = transform(image=image, bboxes=bboxes, class_labels=class_labels)
        transformed_image = transformed['image']
        transformed_bboxes = transformed['bboxes']
        transformed_class_labels = transformed['class_labels']


        augmented_image_path = os.path.join(output_images_dir, f"aug_{i}_{image_file}")
        cv2.imwrite(augmented_image_path, cv2.cvtColor(transformed_image, cv2.COLOR_RGB2BGR))

        augmented_label_path = os.path.join(output_labels_dir, f"aug_{i}_{label_file}")
        with open(augmented_label_path, 'w') as f:
            for bbox, class_id in zip(transformed_bboxes, transformed_class_labels):
                f.write(f"{int(class_id)} {' '.join(map(str, bbox))}\n")

        all_images.append((transformed_image, transformed_bboxes, f"Augmented {i+1}"))

num_images = len(all_images)
num_cols = len(transformations) + 1  
num_rows = len(image_files[:3])  

fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 4, num_rows * 4))

for i in range(num_rows):
    for j in range(num_cols):
        img, bboxes, title = all_images[i * num_cols + j]
        
        for bbox in bboxes:
            x_center, y_center, width, height = bbox
            x_min = int((x_center - width / 2) * img.shape[1])
            y_min = int((y_center - height / 2) * img.shape[0])
            x_max = int((x_center + width / 2) * img.shape[1])
            y_max = int((y_center + height / 2) * img.shape[0])
            cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)
        
        axes[i, j].imshow(img)
        axes[i, j].set_title(title, fontsize=10, fontweight="bold")
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
original_image_count = len([f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))])
print(f"Number of original images: {original_image_count}")

augmented_image_count = len([f for f in os.listdir(output_images_dir) if f.endswith(('.jpg', '.png'))])
print(f"Number of enhanced images: {augmented_image_count}")

total_images = original_image_count + augmented_image_count
print(f"Total number of photos after editing: {total_images}")

In [ ]:
import os
import re
import shutil
from sklearn.model_selection import train_test_split

original_images_dir = '/kaggle/working/yolo_format/images'
augmented_images_dir = '/kaggle/working/yolo_format/augmented_images'
original_labels_dir = '/kaggle/working/yolo_format/labels'
augmented_labels_dir = '/kaggle/working/yolo_format/augmented_labels'

output_dir = '/kaggle/working/dataset'

original_images = sorted(f for f in os.listdir(original_images_dir) if f.endswith(('.jpg', '.png')))
augmented_images = sorted(f for f in os.listdir(augmented_images_dir) if f.endswith(('.jpg', '.png')))
original_labels = [f for f in os.listdir(original_labels_dir) if f.endswith('.txt')]
augmented_labels = [f for f in os.listdir(augmented_labels_dir) if f.endswith('.txt')]
all_labels = original_labels + augmented_labels
def split_originals_with_augmentations(original_images, augmented_images):
    """Split source images first so every augmentation stays with its source."""
    train_originals, val_originals = train_test_split(
        original_images, test_size=0.2, random_state=42
    )
    original_names = set(original_images)
    augmented_sources = {}
    for image_name in augmented_images:
        match = re.fullmatch(r"aug_\d+_(.+)", image_name)
        if match is None or match.group(1) not in original_names:
            raise ValueError(f"Cannot map augmented image to an original: {image_name}")
        augmented_sources[image_name] = match.group(1)

    train_names = set(train_originals)
    val_names = set(val_originals)
    train_images = train_originals + [
        name for name in augmented_images if augmented_sources[name] in train_names
    ]
    val_images = val_originals + [
        name for name in augmented_images if augmented_sources[name] in val_names
    ]
    return train_images, val_images


train_images, val_images = split_originals_with_augmentations(
    original_images, augmented_images
)

train_dir = os.path.join(output_dir, 'images/train')
val_dir = os.path.join(output_dir, 'images/val')
train_labels_dir = os.path.join(output_dir, 'labels/train')
val_labels_dir = os.path.join(output_dir, 'labels/val')

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(train_labels_dir, exist_ok=True)
os.makedirs(val_labels_dir, exist_ok=True)

def copy_files(image_list, src_img_dir1, src_img_dir2, src_lbl_dir1, src_lbl_dir2, dest_img_dir, dest_lbl_dir):
    for image_file in image_list:
        if image_file in original_images:
            src_img = os.path.join(src_img_dir1, image_file)
            src_lbl = os.path.join(src_lbl_dir1, os.path.splitext(image_file)[0] + ".txt")
        else:
            src_img = os.path.join(src_img_dir2, image_file)
            src_lbl = os.path.join(src_lbl_dir2, os.path.splitext(image_file)[0] + ".txt")

        shutil.copy(src_img, os.path.join(dest_img_dir, image_file))

        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, os.path.join(dest_lbl_dir, os.path.basename(src_lbl)))

copy_files(train_images, original_images_dir, augmented_images_dir, original_labels_dir, augmented_labels_dir, train_dir, train_labels_dir)
copy_files(val_images, original_images_dir, augmented_images_dir, original_labels_dir, augmented_labels_dir, val_dir, val_labels_dir)
print(f"The database is successfully equipped: {len(train_images)} Picture of the training and {len(val_images)} Photo for verification.")

In [ ]:
import yaml
yaml_path = '/kaggle/working/dataset.yaml'
dataset_config = {
    "path": "/kaggle/working/dataset",
    "train": "images/train",
    "val": "images/val",
    "nc": num_classes,
    "names": class_names
}

with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False, allow_unicode=True)

print(f"YAML file successfully created: {yaml_path}")

In [ ]:
!pip install ultralytics
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  

model.train(data=yaml_path, epochs=170, imgsz=640, batch=16, name="yolov8_nano_custom")

In [ ]:
model_save_path = "/kaggle/working/runs/detect/yolov8_nano_custom/best.pt"

In [ ]:
metrics = model.val(data=yaml_path)

print(f"Precision: {metrics.results_dict['metrics/precision(B)']}")
print(f"Recall: {metrics.results_dict['metrics/recall(B)']}")
print(f"mAP50: {metrics.results_dict['metrics/mAP50(B)']}")
print(f"mAP50-95: {metrics.results_dict['metrics/mAP50-95(B)']}")

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

precision = metrics.results_dict['metrics/precision(B)']
recall = metrics.results_dict['metrics/recall(B)']
map50 = metrics.results_dict['metrics/mAP50(B)']
map5095 = metrics.results_dict['metrics/mAP50-95(B)']

metrics_names = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
metrics_values = [precision, recall, map50, map5095]

plt.figure(figsize=(8, 5))
plt.bar(metrics_names, metrics_values, color=['blue', 'green', 'orange', 'purple'])
plt.ylim(0, 1)
plt.title("YOLOv8 Model Performance Metrics")
plt.ylabel("Score")
plt.xlabel("Metrics")
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i, v in enumerate(metrics_values):
        plt.text(i, v + 0.02, f"{v:.2f}", ha='center', fontsize=10)

plt.show()

In [ ]:
image_path = '/kaggle/input/weed-detection/test/ridderzuring_3151_jpg.rf.175e23691b2ce606f3054d24896fd639.jpg'  
predictions = model(image_path)
predictions[0].show()
output_image_path = '/kaggle/working/predicted_image.jpg' 
predictions[0].save(output_image_path)

print(f"Predicted image saved to: {output_image_path}")

In [ ]:
for box in predictions[0].boxes:
    print(f"Class: {box.cls}, Confidence: {box.conf}, Coordinates: {box.xywh}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = '/kaggle/working/runs/detect/yolov8_nano_custom/results.csv'

data = pd.read_csv(csv_path)

print(data.head())

plt.figure(figsize=(10, 6))
plt.plot(data['epoch'], data['train/box_loss'], label='Train Box Loss')
plt.plot(data['epoch'], data['val/box_loss'], label='Val Box Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Box Loss')
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(data['epoch'], data['train/cls_loss'], label='Train Class Loss')
plt.plot(data['epoch'], data['val/cls_loss'], label='Val Class Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Class Loss')
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(data['epoch'], data['metrics/mAP50(B)'], label='mAP@50')
plt.xlabel('Epoch')
plt.ylabel('mAP@50')
plt.title('mAP@50')
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(data['epoch'], data['metrics/mAP50-95(B)'], label='mAP@50-95')
plt.xlabel('Epoch')
plt.ylabel('mAP@50-95')
plt.title('mAP@50-95')
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(data['epoch'], data['metrics/precision(B)'], label='Precision')
plt.xlabel('Epoch')
plt.ylabel('Precision')
plt.title('Precision')
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(data['epoch'], data['metrics/recall(B)'], label='Recall')
plt.xlabel('Epoch')
plt.ylabel('Recall')
plt.title('Recall')
plt.legend()
plt.show()